# День 4. Слияние данных из разных источников

**Курс:** ДПК «Практические навыки и опыт анализа данных с помощью Python»  
**Источник данных:** карьер «Восточный» (золотодобыча) — реальная производственная телеметрия и геомеханика.

> 💡 Заполняйте ячейки `# TODO`. Подсказки даны в комментариях. Не стесняйтесь спрашивать преподавателя.


## Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Базовые настройки
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

DATA_DIR = '../datasets'  # путь к папке с датасетами


## Задание 4.1. «Слияние телеметрии бурения с литологией породы»

**Данные:**
- `telemetry_for_merge.csv` — телеметрия без литотипа.
- `lithology_points.csv` — точечные замеры литологии в скважинах (drill_passport, depth_m, litology_code).
- `lithology_intervals.csv` — те же данные, но в виде интервалов «от-до» (для сравнения двух стратегий).

**Задача:** к каждой записи телеметрии привязать литотип на её глубине.


In [ ]:
telem = pd.read_csv(f'{DATA_DIR}/telemetry_for_merge.csv')
litho_points = pd.read_csv(f'{DATA_DIR}/lithology_points.csv')
print('Телеметрия:', telem.shape)
print('Замеры литологии:', litho_points.shape)
litho_points.head()


### Способ 1 (рекомендуется): merge_asof с direction='nearest'

Для каждой записи телеметрии находим **ближайший по глубине** замер литологии внутри той же скважины.

**Важно:** для `merge_asof` обе таблицы должны быть отсортированы по `on`-полю (depth).

In [ ]:
telem_sorted = telem.sort_values('SEGMENT_DEPTH').reset_index(drop=True)
litho_sorted = litho_points.sort_values('depth_m').reset_index(drop=True)

# TODO: применить pd.merge_asof

merged = ...
print('Размер:', merged.shape, 'NaN литотипа:', merged['litology_code'].isnull().sum())
merged[['Drill_Passport', 'SEGMENT_DEPTH', 'depth_m', 'litology_code']].head()


### Сравнение направлений merge_asof

Попробуйте `direction='backward'` и `direction='forward'` — посмотрите, как меняется распределение литотипов:

In [ ]:
for d in ['backward', 'forward', 'nearest']:
    m = pd.merge_asof()
    n_nan = m['litology_code']...
    print(f'direction={d!r}: NaN литотипа = {n_nan} ({n_nan/len(m)*100:.1f}%)')


### Способ 2: интервальное слияние через pd.merge + фильтр

Альтернатива — использовать **интервалы** литологии. Для каждой записи телеметрии находим интервал `[depth_from, depth_to)`, в который она попадает.

In [ ]:
intervals = pd.read_csv(f'{DATA_DIR}/lithology_intervals.csv')
print('Интервалов:', intervals.shape)
intervals.head()


In [ ]:
# TODO: cross-merge по drill_passport, потом фильтрация
merged_classic = ...
print('Записей телеметрии всего:', len(telem))
print('Найдено через интервалы:', len(merged_classic))
print('Покрытие:', f'{len(merged_classic)/len(telem)*100:.1f}%')


### Вопрос на размышление

Какой подход даёт более полное покрытие? Почему?  
В каких ситуациях правильнее использовать **точечный nearest**, а в каких — **интервальный**?


In [ ]:
# Сохраняем результат (используем merged из nearest — у него 100% покрытие)
merged.to_csv('drilling_with_lithology.csv', index=False)
print('Сохранено в drilling_with_lithology.csv')


---
## Задание 4.2. «Полное досье скважины» — многоуровневое слияние

Цепочка из 3 merge:
1. Телеметрия + литология — уже сделано в 4.1.
2. + геомеханика (по литотипу) из `lithology.csv`.
3. + параметры заряда (по drill_passport и hole_number) из `charge_passports.csv`.


In [ ]:
tel_lit = pd.read_csv('drilling_with_lithology.csv')
lithology = pd.read_csv(f'{DATA_DIR}/lithology.csv')
charges = pd.read_csv(f'{DATA_DIR}/charge_passports.csv')
print('telemetry+litho:', tel_lit.shape)
print('lithology:', lithology.shape)
print('charges:', charges.shape)


### Шаг 1. Добавляем геомеханику

In [ ]:
# TODO: merge с lithology.csv по литотипу

step1 = ...
print('После merge с lithology:', step1.shape)


### Шаг 2. Добавляем паспорт заряда

In [ ]:
# TODO: merge с charges по drill_passport + hole_number = Hole_Code
gold = ...
print('Финальный (gold):', gold.shape)
print('Колонок:', len(gold.columns))


In [ ]:
# Сохраняем «золотой» датасет для дней 5-7
gold.to_csv('gold_dataset.csv', index=False)
print('Сохранено в gold_dataset.csv')
